# 03 - Dépôt dans HDFS
Copie les 522 CSV dans le stockage distribué, avec 2 copies de chaque bloc (une par DataNode).


## 1. Ouvrir la SparkSession (pour utiliser son client HDFS)

In [1]:
import time
from pathlib import Path
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("03_depot_hdfs").getOrCreate()
print("SparkSession ouverte :", spark.sparkContext.appName)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 14:12:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/12 14:12:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession ouverte : 03_depot_hdfs


## 2. Se connecter à HDFS avec le client Hadoop du driver

In [2]:
hadoop = spark._jvm.org.apache.hadoop.fs        # les classes Java de Hadoop
conf = spark._jsc.hadoopConfiguration()          # la config lue dans spark-defaults.conf
fs = hadoop.FileSystem.get(conf)                 # le client HDFS

print("Système de fichiers  :", fs.getUri())
print("Réplication demandée :", conf.get("dfs.replication"))

Système de fichiers  : hdfs://namenode:9000
Réplication demandée : 2


## 3. Copier les CSV dans HDFS, année par année


In [3]:
LOCAL = Path("/work/data/landing/seismes")
CIBLE = "/landing/seismes"

fs.mkdirs(hadoop.Path(CIBLE))

t0 = time.time()
for dossier in sorted(LOCAL.iterdir()):
    fichiers = sorted(dossier.glob("*.csv"))
    for f in fichiers:
        source = hadoop.Path("file://" + str(f))                       # fichier local
        destination = hadoop.Path(f"{CIBLE}/{dossier.name}/{f.name}")  # fichier HDFS
        fs.copyFromLocalFile(False, True, source, destination)         # (garder la source, écraser)
    print(f"{dossier.name} : {len(fichiers)} fichiers déposés")

print(f"Durée : {time.time() - t0:.0f} s")

2016 : 53 fichiers déposés
2017 : 52 fichiers déposés
2018 : 52 fichiers déposés
2019 : 52 fichiers déposés
2020 : 52 fichiers déposés
2021 : 53 fichiers déposés
2022 : 52 fichiers déposés
2023 : 52 fichiers déposés
2024 : 52 fichiers déposés
2025 : 52 fichiers déposés
Durée : 62 s


## 4. Vérification : même nombre de fichiers et d'octets en local et dans HDFS

In [4]:
fichiers_locaux = list(LOCAL.glob("*/*.csv"))
octets_local = sum(f.stat().st_size for f in fichiers_locaux)

resume = fs.getContentSummary(hadoop.Path(CIBLE))

print(f"Local : {len(fichiers_locaux)} fichiers, {octets_local} octets")
print(f"HDFS  : {resume.getFileCount()} fichiers, {resume.getLength()} octets")
print("Identique :", len(fichiers_locaux) == resume.getFileCount() and octets_local == resume.getLength())

Local : 522 fichiers, 300320565 octets
HDFS  : 522 fichiers, 300320565 octets
Identique : True


## 5. Vérification : réplication et taille des fichiers (problème des petits fichiers)

In [5]:
statuts = fs.globStatus(hadoop.Path(CIBLE + "/*/*.csv"))
replications = set(s.getReplication() for s in statuts)
taille_moyenne = sum(s.getLen() for s in statuts) / len(statuts)
taille_bloc = statuts[0].getBlockSize()

print("Fichiers dans HDFS     :", len(statuts))
print("Réplications trouvées  :", replications, "(attendu : {2})")
print(f"Taille moyenne fichier : {taille_moyenne / 1e3:.0f} Ko")
print(f"Taille d'un bloc HDFS  : {taille_bloc / 1e6:.0f} Mo")

Fichiers dans HDFS     : 522
Réplications trouvées  : {2} (attendu : {2})
Taille moyenne fichier : 575 Ko
Taille d'un bloc HDFS  : 134 Mo


## 6. Fermer la SparkSession (libère les 4 cœurs pour le notebook suivant)

In [6]:
spark.stop()